In [3]:
!pip install openai pandas --quiet

In [4]:
from google.colab import userdata
import openai
import pandas as pd

openai.api_key = userdata.get('OPENAI_API_KEY')

client = openai.OpenAI(api_key=openai.api_key)
print("OpenAI connected ✓")

OpenAI connected ✓


In [5]:
import pandas as pd
import random
from datetime import datetime, timedelta

random.seed(42)

first_names = ["James","Maria","David","Sarah","Carlos","Aisha","Michael","Priya","Robert","Linda",
               "Kevin","Fatima","Brian","Nicole","Omar","Jessica","Anthony","Yuki","Daniel","Grace",
               "Marcus","Elena","Tyler","Zoe","Andre","Nadia","Joshua","Claire","Malik","Sofia",
               "Derek","Amara","Patrick","Rachel","Victor","Hannah","Jerome","Mei","Raymond","Leila",
               "Curtis","Isabelle","Darnell","Tanya","Felix","Camille","Rodney","Diana","Terrence","Alicia"]

last_names = ["Johnson","Garcia","Williams","Smith","Martinez","Brown","Davis","Wilson","Anderson","Taylor",
              "Thomas","Jackson","White","Harris","Martin","Thompson","Robinson","Clark","Lewis","Walker",
              "Hall","Allen","Young","King","Wright","Scott","Torres","Nguyen","Hill","Adams",
              "Baker","Rivera","Nelson","Mitchell","Carter","Perez","Roberts","Evans","Turner","Phillips",
              "Campbell","Parker","Edwards","Collins","Stewart","Sanchez","Morris","Rogers","Reed","Cook"]

vehicle_types = ["Sedan","SUV","Van","Truck","Minibus"]
license_levels = ["Standard","Commercial","CDL-A","CDL-B","Medical Transport Certified"]
zones = ["North","South","East","West","Central"]
shifts = [("06:00","14:00"),("08:00","16:00"),("10:00","18:00"),("12:00","20:00"),("14:00","22:00")]
shift_names = ["Early","Morning","Midday","Afternoon","Evening"]

drivers = []
for i in range(50):
    shift_idx = i % 5
    s_start, s_end = shifts[shift_idx]
    drivers.append({
        "driver_id": f"DRV{str(i+1).zfill(3)}",
        "name": f"{first_names[i]} {last_names[i]}",
        "phone": f"+1555{random.randint(1000000,9999999)}",
        "vehicle_type": vehicle_types[i % 5],
        "license_level": random.choice(license_levels),
        "zone": zones[i % 5],
        "shift": shift_names[shift_idx],
        "availability_start": s_start,
        "availability_end": s_end,
        "status": "available",
        "trips_completed_today": 0
    })

drivers_df = pd.DataFrame(drivers)

origins = ["123 Main St","456 Oak Ave","789 Pine Rd","321 Elm St","654 Maple Dr",
           "987 Cedar Ln","147 Birch Blvd","258 Walnut Way","369 Spruce Ct","741 Willow Pl",
           "852 Chestnut Ave","963 Poplar Rd","111 Ash St","222 Hickory Dr","333 Sycamore Ln",
           "Airport Terminal A","Airport Terminal B","Central Station","North Bus Depot","South Ferry"]

destinations = ["City Hospital","Downtown Office Complex","Tech Park Campus","University Campus",
                "Shopping Center","Convention Center","Sports Arena","Hotel Grand","Airport Terminal A",
                "Airport Terminal B","Central Station","Medical Center","Government Building",
                "Research Institute","Financial District","Waterfront Plaza","Industrial Park",
                "Residential Complex A","Residential Complex B","Community Center"]

trip_types = ["Corporate shuttle","Medical transport","Delivery run","Airport transfer",
              "Campus shuttle","Event transport","Emergency supply run","VIP transfer"]

priorities = ["Low","Normal","Normal","Normal","High","Urgent"]

trips = []
for i in range(100):
    shift_idx = i % 5
    s_start, s_end = shifts[shift_idx]
    start_hour = int(s_start.split(":")[0])
    pickup_dt = datetime(2025, 3, 20, start_hour, 0) + timedelta(minutes=random.randint(30, 180))
    duration = random.choice([20, 30, 45, 60])
    trips.append({
        "trip_id": f"TRP{str(i+1).zfill(3)}",
        "trip_type": random.choice(trip_types),
        "pickup_time": pickup_dt.strftime("%H:%M"),
        "pickup_date": pickup_dt.strftime("%Y-%m-%d"),
        "duration_mins": duration,
        "origin": random.choice(origins),
        "destination": random.choice(destinations),
        "vehicle_required": vehicle_types[i % 5],
        "zone": zones[i % 5],
        "passengers": random.randint(1, 8),
        "priority": random.choice(priorities),
        "special_requirements": random.choice(["None","Wheelchair accessible","Temperature controlled","None","None"]),
        "status": "unassigned",
        "assigned_driver_id": ""
    })

trips_df = pd.DataFrame(trips)

print("drivers.csv:", drivers_df.shape)
print("trips.csv:", trips_df.shape)
print("Driver statuses:", drivers_df["status"].value_counts().to_dict())

drivers.csv: (50, 11)
trips.csv: (100, 14)
Driver statuses: {'available': 50}


In [6]:
drivers_df = pd.read_csv('drivers.csv')
trips_df = pd.read_csv('trips.csv')

In [7]:
print(drivers_df.shape)
print(trips_df.shape)
print(drivers_df.head(3))
print(trips_df.head(3))

(50, 11)
(100, 14)
  driver_id            name        phone vehicle_type  \
0    DRV001   James Johnson  15551419610          Van   
1    DRV002    Maria Garcia  15552458591      Minibus   
2    DRV003  David Williams  15554903402      Minibus   

                 license_level   zone      shift availability_start  \
0                   Commercial  South    Morning              06:00   
1                        CDL-B  North     Midday              10:00   
2  Medical Transport Certified  North  Afternoon              14:00   

  availability_end     status  trips_completed_today  
0            14:00  available                      1  
1            18:00  available                      1  
2            22:00       busy                      3  
  trip_id        trip_type pickup_time pickup_date  duration_mins  \
0  TRP001   Campus shuttle       09:28  2025-03-20             90   
1  TRP002  Event transport       06:45  2025-03-20             20   
2  TRP003     Delivery run       07:17  

In [8]:
from datetime import datetime, timedelta

def trips_overlap(trip1, trip2):
    try:
        start1 = datetime.strptime(trip1['pickup_time'], "%H:%M")
        end1 = start1 + timedelta(minutes=int(trip1['duration_mins']))
        start2 = datetime.strptime(trip2['pickup_time'], "%H:%M")
        end2 = start2 + timedelta(minutes=int(trip2['duration_mins']))
        return not (end1 <= start2 or end2 <= start1)
    except:
        return True

def update_status(driver_id, trip_id, drivers_df, trips_df):
    assigned_trips = trips_df[
        (trips_df['assigned_driver_id'] == driver_id) &
        (trips_df['trip_id'] != trip_id)
    ]

    current_trip = trips_df[trips_df['trip_id'] == trip_id].iloc[0]

    has_conflict = any(
        trips_overlap(current_trip, t)
        for _, t in assigned_trips.iterrows()
    )

    if not has_conflict:
        trips_df.loc[trips_df['trip_id'] == trip_id, 'status'] = 'assigned'
        trips_df.loc[trips_df['trip_id'] == trip_id, 'assigned_driver_id'] = driver_id
        trip_count = len(assigned_trips) + 1
        drivers_df.loc[drivers_df['driver_id'] == driver_id, 'trips_completed_today'] = trip_count

    return drivers_df, trips_df

print("Status updater ready.")

Status updater ready.


In [9]:
def gpt_match_driver(trip, drivers_df):
    candidates = drivers_df[
        (drivers_df['status'] == 'available') &
        (drivers_df['vehicle_type'] == trip['vehicle_required']) &
        (drivers_df['zone'] == trip['zone'])
    ].copy()

    if candidates.empty:
        candidates = drivers_df[
            (drivers_df['status'] == 'available') &
            (drivers_df['vehicle_type'] == trip['vehicle_required'])
        ].copy()

    if candidates.empty:
        candidates = drivers_df[
            (drivers_df['status'] == 'available')
        ].copy()

    if candidates.empty:
        return None, "No available drivers found for this trip."

    candidates = candidates[
        candidates.apply(lambda d: is_available_for_trip(d, trip), axis=1)
    ]

    if candidates.empty:
        return None, "No drivers available within the required time window."

    drivers_context = candidates[[
        'driver_id', 'name', 'vehicle_type', 'license_level',
        'zone', 'shift', 'availability_start', 'availability_end',
        'trips_completed_today'
    ]].to_string(index=False)

    prompt = f"""You are a trip assignment system. Select the single best driver for this trip.

TRIP DETAILS:
- Trip ID: {trip['trip_id']}
- Type: {trip['trip_type']}
- Pickup time: {trip['pickup_time']}
- Duration: {trip['duration_mins']} minutes
- From: {trip['origin']}
- To: {trip['destination']}
- Vehicle required: {trip['vehicle_required']}
- Zone: {trip['zone']}
- Passengers: {trip['passengers']}
- Priority: {trip['priority']}
- Special requirements: {trip['special_requirements']}

AVAILABLE DRIVERS:
{drivers_context}

SELECTION RULES:
1. Driver must be in the correct zone if possible
2. Vehicle type must match
3. Prefer drivers with fewer trips_completed_today
4. For Urgent/High priority, prefer CDL-A or certified drivers
5. For special requirements, prefer certified drivers

Respond in this exact format:
DRIVER_ID: <driver_id>
REASON: <one sentence explanation>"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    result = response.choices[0].message.content.strip()
    driver_id_match = re.search(r'DRIVER_ID:\s*(\w+)', result)
    reason_match = re.search(r'REASON:\s*(.+)', result)

    if driver_id_match:
        driver_id = driver_id_match.group(1)
        reason = reason_match.group(1) if reason_match else "No reason provided"
        return driver_id, reason
    else:
        return None, "GPT response could not be parsed."

print("GPT matching engine ready.")

GPT matching engine ready.


In [10]:
from datetime import datetime

def update_status(driver_id, trip_id, drivers_df, trips_df):
    drivers_df.loc[drivers_df['driver_id'] == driver_id, 'status'] = 'busy'
    trips_df.loc[trips_df['trip_id'] == trip_id, 'status'] = 'assigned'
    trips_df.loc[trips_df['trip_id'] == trip_id, 'assigned_driver_id'] = driver_id
    return drivers_df, trips_df

print("Status updater ready.")

Status updater ready.


In [11]:
def notify_driver(driver_id, trip, drivers_df):
    driver = drivers_df[drivers_df['driver_id'] == driver_id]

    if driver.empty:
        print(f"NOTIFICATION WARNING: Driver {driver_id} not found.")
        return

    driver = driver.iloc[0]

    print("=" * 50)
    print(f"NOTIFICATION SENT TO: {driver['name']}")
    print(f"Phone: {driver['phone']}")
    print("-" * 50)
    print(f"New trip assignment: {trip['trip_id']}")
    print(f"Type: {trip['trip_type']}")
    print(f"Pickup time: {trip['pickup_time']}")
    print(f"From: {trip['origin']}")
    print(f"To: {trip['destination']}")
    print(f"Passengers: {trip['passengers']}")
    print(f"Priority: {trip['priority']}")
    print(f"Special requirements: {trip['special_requirements']}")
    print("=" * 50)

print("Notification system ready.")

Notification system ready.


In [12]:
import csv
import os

def log_assignment(driver_id, trip_id, reason, drivers_df, trips_df):
    driver = drivers_df[drivers_df['driver_id'] == driver_id]
    trip = trips_df[trips_df['trip_id'] == trip_id]

    if driver.empty or trip.empty:
        print(f"LOG WARNING: Could not find driver {driver_id} or trip {trip_id}")
        return

    driver = driver.iloc[0]
    trip = trip.iloc[0]

    log_entry = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'trip_id': trip_id,
        'driver_id': driver_id,
        'driver_name': driver['name'],
        'trip_type': trip['trip_type'],
        'pickup_time': trip['pickup_time'],
        'origin': trip['origin'],
        'destination': trip['destination'],
        'priority': trip['priority'],
        'gpt_reasoning': reason
    }

    log_file = 'assignment_log.csv'
    file_exists = os.path.isfile(log_file)

    with open(log_file, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=log_entry.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(log_entry)

print("Assignment logger ready.")

Assignment logger ready.


In [13]:
def run_assignments(trips_df, drivers_df):
    unassigned = trips_df[trips_df['status'] == 'unassigned'].copy()

    total = len(unassigned)
    assigned_count = 0
    failed_count = 0

    print(f"Processing {total} unassigned trips...\n")

    for _, trip in unassigned.iterrows():
        driver_id, reason = gpt_match_driver(trip, drivers_df)

        if driver_id:
            drivers_df, trips_df = update_status(driver_id, trip['trip_id'], drivers_df, trips_df)
            notify_driver(driver_id, trip, drivers_df)
            log_assignment(driver_id, trip['trip_id'], reason, drivers_df, trips_df)
            assigned_count += 1
        else:
            print(f"UNASSIGNED: {trip['trip_id']} - {reason}")
            failed_count += 1

    print("\n" + "=" * 50)
    print("ASSIGNMENT SUMMARY")
    print("=" * 50)
    print(f"Total trips processed: {total}")
    print(f"Successfully assigned: {assigned_count}")
    print(f"Could not assign: {failed_count}")
    print(f"Assignment rate: {round(assigned_count/total*100, 1)}%")
    print("=" * 50)

    return drivers_df, trips_df

print("Orchestration loop ready.")

Orchestration loop ready.


In [14]:
drivers_df = pd.read_csv('drivers.csv')
trips_df = pd.read_csv('trips.csv')
print("Data reloaded.")

Data reloaded.


In [15]:
import re
from datetime import datetime, timedelta

def is_available_for_trip(driver, trip):
    driver_start_str = driver['availability_start']
    driver_end_str = driver['availability_end']
    trip_pickup_str = trip['pickup_time']
    trip_duration = trip['duration_mins']

    driver_start = datetime.strptime(driver_start_str, "%H:%M")
    driver_end = datetime.strptime(driver_end_str, "%H:%M")
    trip_pickup = datetime.strptime(trip_pickup_str, "%H:%M")
    trip_end = trip_pickup + timedelta(minutes=int(trip_duration))

    # Check if trip falls within driver's availability window
    return driver_start <= trip_pickup and trip_end <= driver_end

def gpt_match_driver(trip, drivers_df):
    candidates = drivers_df[
        (drivers_df['status'] == 'available') &
        (drivers_df['vehicle_type'] == trip['vehicle_required']) &
        (drivers_df['zone'] == trip['zone'])
    ].copy()

    if candidates.empty:
        candidates = drivers_df[
            (drivers_df['status'] == 'available') &
            (drivers_df['vehicle_type'] == trip['vehicle_required'])
        ].copy()

    if candidates.empty:
        candidates = drivers_df[
            (drivers_df['status'] == 'available')
        ].copy()

    if candidates.empty:
        return None, "No available drivers found for this trip."

    candidates = candidates[
        candidates.apply(lambda d: is_available_for_trip(d, trip), axis=1)
    ]

    if candidates.empty:
        return None, "No drivers available within the required time window."

    drivers_context = candidates[[
        'driver_id', 'name', 'vehicle_type', 'license_level',
        'zone', 'shift', 'availability_start', 'availability_end',
        'trips_completed_today'
    ]].to_string(index=False)

    prompt = f"""You are a trip assignment system. Select the single best driver for this trip.

TRIP DETAILS:
- Trip ID: {trip['trip_id']}
- Type: {trip['trip_type']}
- Pickup time: {trip['pickup_time']}
- Duration: {trip['duration_mins']} minutes
- From: {trip['origin']}
- To: {trip['destination']}
- Vehicle required: {trip['vehicle_required']}
- Zone: {trip['zone']}
- Passengers: {trip['passengers']}
- Priority: {trip['priority']}
- Special requirements: {trip['special_requirements']}

AVAILABLE DRIVERS:
{drivers_context}

SELECTION RULES:
1. Driver must be in the correct zone if possible
2. Vehicle type must match
3. Prefer drivers with fewer trips_completed_today
4. For Urgent/High priority, prefer CDL-A or certified drivers
5. For special requirements, prefer certified drivers

Respond in this exact format:
DRIVER_ID: <driver_id>
REASON: <one sentence explanation>"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    result = response.choices[0].message.content.strip()
    driver_id_match = re.search(r'DRIVER_ID:\s*(\w+)', result)
    reason_match = re.search(r'REASON:\s*(.+)', result)

    if driver_id_match:
        driver_id = driver_id_match.group(1)
        reason = reason_match.group(1) if reason_match else "No reason provided"
        return driver_id, reason
    else:
        return None, "GPT response could not be parsed."

drivers_df, trips_df = run_assignments(trips_df, drivers_df)

Processing 76 unassigned trips...

NOTIFICATION SENT TO: Tyler Young
Phone: 15557089806
--------------------------------------------------
New trip assignment: TRP001
Type: Campus shuttle
Pickup time: 09:28
From: Airport Terminal B
To: Waterfront Plaza
Passengers: 2
Priority: Urgent
Special requirements: Security clearance
NOTIFICATION SENT TO: Hannah Perez
Phone: 15552564689
--------------------------------------------------
New trip assignment: TRP002
Type: Event transport
Pickup time: 06:45
From: 654 Maple Dr
To: Airport Terminal A
Passengers: 7
Priority: High
Special requirements: nan
UNASSIGNED: TRP003 - No drivers available within the required time window.
UNASSIGNED: TRP005 - No drivers available within the required time window.
UNASSIGNED: TRP006 - No drivers available within the required time window.
UNASSIGNED: TRP007 - No drivers available within the required time window.
NOTIFICATION SENT TO: Curtis Campbell
Phone: 15553641282
-----------------------------------------------

In [19]:
def manager_portal():
    global trips_df, drivers_df

    print("=" * 60)
    print("MANAGER PORTAL")
    print("=" * 60)

    print("\n--- TRIP ASSIGNMENT STATUS ---\n")
    summary = trips_df[['trip_id', 'trip_type', 'pickup_time',
                         'priority', 'status', 'assigned_driver_id']]
    print(summary.to_string(index=False))

    print("\n--- ASSIGNMENT STATS ---\n")
    total = len(trips_df)
    assigned = len(trips_df[trips_df['status'] == 'assigned'])
    unassigned = len(trips_df[trips_df['status'] == 'unassigned'])
    print(f"Total trips:      {total}")
    print(f"Assigned:         {assigned}")
    print(f"Unassigned:       {unassigned}")
    print(f"Assignment rate:  {round(assigned/total*100, 1)}%")

    print("\n--- DRIVER WORKLOAD ---\n")
    workload = drivers_df[['driver_id', 'name', 'vehicle_type',
                            'zone', 'status', 'trips_completed_today']]
    print(workload.to_string(index=False))

    print("\n--- CREATE NEW TRIP ---\n")
    trip_id = f"TRP{str(len(trips_df)+1).zfill(3)}"
    trip_type = input("Trip type: ")
    pickup_time = input("Pickup time (HH:MM): ")
    duration = input("Duration (minutes): ")
    origin = input("Origin: ")
    destination = input("Destination: ")
    vehicle = input("Vehicle required (Sedan/SUV/Van/Truck/Minibus): ")
    zone = input("Zone (North/South/East/West/Central): ")
    passengers = input("Number of passengers: ")
    priority = input("Priority (Low/Normal/High/Urgent): ")
    special = input("Special requirements (or None): ")

    new_trip = {
        "trip_id": trip_id,
        "trip_type": trip_type,
        "pickup_time": pickup_time,
        "pickup_date": "2025-03-20",
        "duration_mins": int(duration),
        "origin": origin,
        "destination": destination,
        "vehicle_required": vehicle,
        "zone": zone,
        "passengers": int(passengers),
        "priority": priority,
        "special_requirements": special,
        "status": "unassigned",
        "assigned_driver_id": ""
    }

    trips_df = pd.concat([trips_df, pd.DataFrame([new_trip])], ignore_index=True)

    print(f"\nTrip {trip_id} created successfully.")
    print("Running auto-assignment...\n")

    drivers_df, trips_df = run_assignments(
        trips_df[trips_df['trip_id'] == trip_id], drivers_df
    )

    print("=" * 60)

manager_portal()

MANAGER PORTAL

--- TRIP ASSIGNMENT STATUS ---

trip_id            trip_type pickup_time priority     status assigned_driver_id
 TRP001       Campus shuttle       09:28   Urgent   assigned             DRV023
 TRP002      Event transport       06:45     High   assigned             DRV036
 TRP003         Delivery run       07:17   Normal unassigned                NaN
 TRP004    Corporate shuttle       11:15     High   assigned             DRV049
 TRP005         Delivery run       16:35   Normal unassigned                NaN
 TRP006     Airport transfer       13:01   Normal unassigned                NaN
 TRP007       Campus shuttle       13:51   Normal unassigned                NaN
 TRP008 Emergency supply run       10:45   Normal   assigned             DRV041
 TRP009 Emergency supply run       06:39   Normal   assigned             DRV006
 TRP010 Emergency supply run       10:20   Normal   assigned             DRV011
 TRP011    Medical transport       16:38   Normal   assigned            

In [20]:
def driver_portal():
    global trips_df, drivers_df

    print("=" * 60)
    print("DRIVER PORTAL")
    print("=" * 60)

    driver_id = input("Enter your driver ID: ")

    driver = drivers_df[drivers_df['driver_id'] == driver_id]

    if driver.empty:
        print(f"Driver {driver_id} not found.")
        return

    driver = driver.iloc[0]

    print(f"\nWelcome, {driver['name']}!")
    print(f"Vehicle: {driver['vehicle_type']} | Zone: {driver['zone']} | Shift: {driver['shift']}")
    print(f"Availability: {driver['availability_start']} - {driver['availability_end']}")

    print("\n--- YOUR ASSIGNED TRIPS ---\n")

    my_trips = trips_df[trips_df['assigned_driver_id'] == driver_id]

    if my_trips.empty:
        print("No trips assigned to you.")
        return

    print(my_trips[['trip_id', 'trip_type', 'pickup_time', 'origin',
                     'destination', 'passengers', 'priority',
                     'special_requirements', 'status']].to_string(index=False))

    print("\n--- UPDATE TRIP STATUS ---\n")

    trip_id = input("Enter trip ID to update (or press Enter to skip): ")

    if trip_id:
        valid_trip = my_trips[my_trips['trip_id'] == trip_id]

        if valid_trip.empty:
            print("Trip not found or not assigned to you.")
            return

        print("Status options: en_route / completed / cancelled")
        new_status = input("Enter new status: ")

        trips_df.loc[trips_df['trip_id'] == trip_id, 'status'] = new_status
        print(f"\nTrip {trip_id} status updated to: {new_status}")

        if new_status == 'completed':
            drivers_df.loc[drivers_df['driver_id'] == driver_id, 'status'] = 'available'
            print(f"{driver['name']} is now available for new assignments.")

    print("=" * 60)

driver_portal()

DRIVER PORTAL
Enter your driver ID: DRV006

Welcome, Aisha Brown!
Vehicle: Truck | Zone: East | Shift: Morning
Availability: 06:00 - 14:00

--- YOUR ASSIGNED TRIPS ---

No trips assigned to you.


In [21]:
print(trips_df[trips_df['assigned_driver_id'] == 'DRV006'])

Empty DataFrame
Columns: [trip_id, trip_type, pickup_time, pickup_date, duration_mins, origin, destination, vehicle_required, zone, passengers, priority, special_requirements, status, assigned_driver_id]
Index: []


In [22]:
print(trips_df['status'].value_counts())
print(trips_df['assigned_driver_id'].value_counts().head(10))

status
unassigned    1
Name: count, dtype: int64
assigned_driver_id
    1
Name: count, dtype: int64


In [23]:
drivers_df = pd.read_csv('drivers.csv')
trips_df = pd.read_csv('trips.csv')
print("Data reloaded.")

Data reloaded.


In [24]:
drivers_df, trips_df = run_assignments(trips_df, drivers_df)

Processing 76 unassigned trips...

NOTIFICATION SENT TO: Tyler Young
Phone: 15557089806
--------------------------------------------------
New trip assignment: TRP001
Type: Campus shuttle
Pickup time: 09:28
From: Airport Terminal B
To: Waterfront Plaza
Passengers: 2
Priority: Urgent
Special requirements: Security clearance
NOTIFICATION SENT TO: Hannah Perez
Phone: 15552564689
--------------------------------------------------
New trip assignment: TRP002
Type: Event transport
Pickup time: 06:45
From: 654 Maple Dr
To: Airport Terminal A
Passengers: 7
Priority: High
Special requirements: nan
UNASSIGNED: TRP003 - No drivers available within the required time window.
UNASSIGNED: TRP005 - No drivers available within the required time window.
UNASSIGNED: TRP006 - No drivers available within the required time window.
UNASSIGNED: TRP007 - No drivers available within the required time window.
NOTIFICATION SENT TO: Curtis Campbell
Phone: 15553641282
-----------------------------------------------

In [25]:
print(trips_df['status'].value_counts())
print(trips_df[trips_df['assigned_driver_id'] == 'DRV006'])

status
assigned      61
unassigned    39
Name: count, dtype: int64
  trip_id             trip_type pickup_time pickup_date  duration_mins  \
8  TRP009  Emergency supply run       06:39  2025-03-20             20   

          origin      destination vehicle_required     zone  passengers  \
8  963 Poplar Rd  Central Station            Truck  Central           2   

  priority special_requirements    status assigned_driver_id  
8   Normal                  NaN  assigned             DRV006  


In [26]:
driver_portal()

DRIVER PORTAL
Enter your driver ID: DRV006

Welcome, Aisha Brown!
Vehicle: Truck | Zone: East | Shift: Morning
Availability: 06:00 - 14:00

--- YOUR ASSIGNED TRIPS ---

trip_id            trip_type pickup_time        origin     destination  passengers priority special_requirements   status
 TRP009 Emergency supply run       06:39 963 Poplar Rd Central Station           2   Normal                  NaN assigned

--- UPDATE TRIP STATUS ---

Enter trip ID to update (or press Enter to skip): 


In [27]:
driver_portal()

DRIVER PORTAL
Enter your driver ID: DRV006

Welcome, Aisha Brown!
Vehicle: Truck | Zone: East | Shift: Morning
Availability: 06:00 - 14:00

--- YOUR ASSIGNED TRIPS ---

trip_id            trip_type pickup_time        origin     destination  passengers priority special_requirements   status
 TRP009 Emergency supply run       06:39 963 Poplar Rd Central Station           2   Normal                  NaN assigned

--- UPDATE TRIP STATUS ---

Enter trip ID to update (or press Enter to skip): 


In [28]:
!pip install flask flask-ngrok pyngrok --quiet

In [29]:
from pyngrok import ngrok
from flask import Flask, request, jsonify, render_template_string
import threading

app = Flask(__name__)

In [30]:
MANAGER_HTML = '''
<!DOCTYPE html>
<html>
<head>
    <title>Manager Portal</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }
        h1 { color: #333; }
        table { border-collapse: collapse; width: 100%; background: white; }
        th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
        th { background: #4CAF50; color: white; }
        tr:nth-child(even) { background: #f9f9f9; }
        .form-section { background: white; padding: 20px; margin-top: 20px; border-radius: 8px; }
        input, select { padding: 8px; margin: 5px; width: 200px; }
        button { padding: 10px 20px; background: #4CAF50; color: white; border: none; cursor: pointer; border-radius: 4px; }
        .stats { display: flex; gap: 20px; margin: 20px 0; }
        .stat-card { background: white; padding: 20px; border-radius: 8px; text-align: center; flex: 1; }
        .stat-card h2 { margin: 0; color: #4CAF50; font-size: 36px; }
    </style>
</head>
<body>
    <h1>Manager Portal</h1>
    <div class="stats">
        <div class="stat-card"><h2>{{ total }}</h2><p>Total Trips</p></div>
        <div class="stat-card"><h2>{{ assigned }}</h2><p>Assigned</p></div>
        <div class="stat-card"><h2>{{ unassigned }}</h2><p>Unassigned</p></div>
        <div class="stat-card"><h2>{{ rate }}%</h2><p>Assignment Rate</p></div>
    </div>
    <h2>Trip Assignment Status</h2>
    <table>
        <tr><th>Trip ID</th><th>Type</th><th>Pickup Time</th><th>Priority</th><th>Status</th><th>Assigned Driver</th></tr>
        {% for trip in trips %}
        <tr>
            <td>{{ trip.trip_id }}</td>
            <td>{{ trip.trip_type }}</td>
            <td>{{ trip.pickup_time }}</td>
            <td>{{ trip.priority }}</td>
            <td>{{ trip.status }}</td>
            <td>{{ trip.assigned_driver_id }}</td>
        </tr>
        {% endfor %}
    </table>
    <div class="form-section">
        <h2>Create New Trip</h2>
        <form action="/create_trip" method="POST">
            <input name="trip_type" placeholder="Trip type" required><br>
            <input name="pickup_time" placeholder="Pickup time (HH:MM)" required><br>
            <input name="duration_mins" placeholder="Duration (minutes)" required><br>
            <input name="origin" placeholder="Origin" required><br>
            <input name="destination" placeholder="Destination" required><br>
            <select name="vehicle_required">
                <option>Sedan</option><option>SUV</option>
                <option>Van</option><option>Truck</option><option>Minibus</option>
            </select><br>
            <select name="zone">
                <option>North</option><option>South</option>
                <option>East</option><option>West</option><option>Central</option>
            </select><br>
            <input name="passengers" placeholder="Passengers" required><br>
            <select name="priority">
                <option>Low</option><option>Normal</option>
                <option>High</option><option>Urgent</option>
            </select><br>
            <input name="special_requirements" placeholder="Special requirements"><br>
            <button type="submit">Create and Auto-Assign</button>
        </form>
    </div>
</body>
</html>
'''

DRIVER_HTML = '''
<!DOCTYPE html>
<html>
<head>
    <title>Driver Portal</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }
        h1 { color: #333; }
        table { border-collapse: collapse; width: 100%; background: white; }
        th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
        th { background: #2196F3; color: white; }
        tr:nth-child(even) { background: #f9f9f9; }
        .form-section { background: white; padding: 20px; margin-top: 20px; border-radius: 8px; }
        input, select { padding: 8px; margin: 5px; width: 200px; }
        button { padding: 10px 20px; background: #2196F3; color: white; border: none; cursor: pointer; border-radius: 4px; }
        .info-card { background: white; padding: 20px; border-radius: 8px; margin-bottom: 20px; }
    </style>
</head>
<body>
    <h1>Driver Portal</h1>
    {% if driver %}
    <div class="info-card">
        <h2>Welcome, {{ driver.name }}!</h2>
        <p>Vehicle: {{ driver.vehicle_type }} | Zone: {{ driver.zone }} | Shift: {{ driver.shift }}</p>
        <p>Availability: {{ driver.availability_start }} - {{ driver.availability_end }}</p>
    </div>
    <h2>Your Assigned Trips</h2>
    {% if trips %}
    <table>
        <tr><th>Trip ID</th><th>Type</th><th>Pickup Time</th><th>From</th><th>To</th><th>Passengers</th><th>Priority</th><th>Status</th></tr>
        {% for trip in trips %}
        <tr>
            <td>{{ trip.trip_id }}</td>
            <td>{{ trip.trip_type }}</td>
            <td>{{ trip.pickup_time }}</td>
            <td>{{ trip.origin }}</td>
            <td>{{ trip.destination }}</td>
            <td>{{ trip.passengers }}</td>
            <td>{{ trip.priority }}</td>
            <td>{{ trip.status }}</td>
        </tr>
        {% endfor %}
    </table>
    <div class="form-section">
        <h2>Update Trip Status</h2>
        <form action="/update_trip" method="POST">
            <input name="driver_id" value="{{ driver.driver_id }}" type="hidden">
            <input name="trip_id" placeholder="Trip ID" required><br>
            <select name="new_status">
                <option>en_route</option>
                <option>completed</option>
                <option>cancelled</option>
            </select><br>
            <button type="submit">Update Status</button>
        </form>
    </div>
    {% else %}
    <p>No trips assigned to you.</p>
    {% endif %}
    {% else %}
    <div class="form-section">
        <h2>Login</h2>
        <form action="/driver" method="POST">
            <input name="driver_id" placeholder="Enter your Driver ID" required><br>
            <button type="submit">Login</button>
        </form>
    </div>
    {% endif %}
</body>
</html>
'''

print("Templates ready.")

Templates ready.


In [31]:
@app.route('/manager')
def manager():
    global trips_df, drivers_df
    trips = trips_df.to_dict('records')
    total = len(trips_df)
    assigned = len(trips_df[trips_df['status'] == 'assigned'])
    unassigned = len(trips_df[trips_df['status'] == 'unassigned'])
    rate = round(assigned / total * 100, 1)
    return render_template_string(MANAGER_HTML, trips=trips,
                                  total=total, assigned=assigned,
                                  unassigned=unassigned, rate=rate)

@app.route('/create_trip', methods=['POST'])
def create_trip():
    global trips_df, drivers_df
    trip_id = f"TRP{str(len(trips_df)+1).zfill(3)}"
    new_trip = {
        'trip_id': trip_id,
        'trip_type': request.form['trip_type'],
        'pickup_time': request.form['pickup_time'],
        'pickup_date': '2025-03-20',
        'duration_mins': int(request.form['duration_mins']),
        'origin': request.form['origin'],
        'destination': request.form['destination'],
        'vehicle_required': request.form['vehicle_required'],
        'zone': request.form['zone'],
        'passengers': int(request.form['passengers']),
        'priority': request.form['priority'],
        'special_requirements': request.form['special_requirements'],
        'status': 'unassigned',
        'assigned_driver_id': ''
    }
    trips_df = pd.concat([trips_df, pd.DataFrame([new_trip])], ignore_index=True)
    drivers_df, trips_df = run_assignments(
        trips_df[trips_df['trip_id'] == trip_id], drivers_df
    )
    return manager()

@app.route('/driver', methods=['GET', 'POST'])
def driver():
    global trips_df, drivers_df
    if request.method == 'POST':
        driver_id = request.form['driver_id']
        driver = drivers_df[drivers_df['driver_id'] == driver_id]
        if driver.empty:
            return render_template_string(DRIVER_HTML, driver=None, trips=None)
        driver = driver.iloc[0].to_dict()
        trips = trips_df[trips_df['assigned_driver_id'] == driver_id].to_dict('records')
        return render_template_string(DRIVER_HTML, driver=driver, trips=trips)
    return render_template_string(DRIVER_HTML, driver=None, trips=None)

@app.route('/update_trip', methods=['POST'])
def update_trip():
    global trips_df, drivers_df
    trip_id = request.form['trip_id']
    driver_id = request.form['driver_id']
    new_status = request.form['new_status']
    trips_df.loc[trips_df['trip_id'] == trip_id, 'status'] = new_status
    if new_status == 'completed':
        drivers_df.loc[drivers_df['driver_id'] == driver_id, 'status'] = 'available'
    driver = drivers_df[drivers_df['driver_id'] == driver_id].iloc[0].to_dict()
    trips = trips_df[trips_df['assigned_driver_id'] == driver_id].to_dict('records')
    return render_template_string(DRIVER_HTML, driver=driver, trips=trips)

print("Routes ready.")

Routes ready.


In [32]:
from pyngrok import ngrok
import threading

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

public_url = ngrok.connect(5000)
print(f"Manager Portal: {public_url}/manager")
print(f"Driver Portal: {public_url}/driver")

threading.Thread(target=app.run, kwargs={'port': 5000}).start()

Manager Portal: NgrokTunnel: "https://isodomic-kayden-decussately.ngrok-free.dev" -> "http://localhost:5000"/manager
Driver Portal: NgrokTunnel: "https://isodomic-kayden-decussately.ngrok-free.dev" -> "http://localhost:5000"/driver
